# Lab 18 — Calibración de Pulsos y Oscilaciones Rabi

Las **oscilaciones de Rabi** son la respuesta periódica de un qubit a una señal resonante.
Al barrer la amplitud del pulso, podemos calibrar cuánta amplitud corresponde a una puerta X (π-pulso).

**Física**: un pulso resonante de área θ = Ωt rota el qubit en un ángulo θ alrededor de un eje en el plano XY.

## 1. Modelo de Rabi

En la aproximación de onda giratoria (RWA), la evolución bajo un pulso resonante es:

$$U(\theta) = \begin{pmatrix} \cos(\theta/2) & -i\sin(\theta/2) \\ -i\sin(\theta/2) & \cos(\theta/2) \end{pmatrix} = R_x(\theta)$$

La probabilidad de medir |1⟩ desde |0⟩ es:

$$P(|1\rangle) = \sin^2(\theta/2)$$

donde θ = Ω·A·t (Ω: frecuencia Rabi, A: amplitud del pulso, t: duración).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.primitives import StatevectorSampler

def rabi_probability(amp: float) -> float:
    """P(|1⟩) = sin²(amp/2) para una oscilación Rabi teórica."""
    return np.sin(amp / 2)**2

# Barrido de amplitud
amplitudes = np.linspace(0, 4 * np.pi, 200)
p1_theory = [rabi_probability(a) for a in amplitudes]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(amplitudes / np.pi, p1_theory, 'b-', linewidth=2)
ax.axhline(1.0, color='gray', linestyle=':', alpha=0.5)
for k in [1, 2, 3, 4]:
    ax.axvline(k, color='red', linestyle='--', alpha=0.4)
    ax.text(k + 0.05, 0.5, f'θ={k}π', color='red', fontsize=9)
ax.set_xlabel('Amplitud θ / π', fontsize=12)
ax.set_ylabel('P(|1⟩)', fontsize=12)
ax.set_title('Oscilación de Rabi teórica: P(|1⟩) = sin²(θ/2)', fontsize=13)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('Calibración de puertas:')
print(f'  X (π-pulso):   θ = π   → P(|1⟩) = {rabi_probability(np.pi):.4f}')
print(f'  √X (π/2-pulso): θ = π/2 → P(|1⟩) = {rabi_probability(np.pi/2):.4f}')

## 2. Simulación con Qiskit: barrido de amplitud

Usamos la puerta `Rx(θ)` de Qiskit para simular el efecto de un pulso resonante.
El ángulo θ es directamente la amplitud del pulso en unidades de π-radianes.

In [ ]:
sampler = StatevectorSampler()

# Barrido de amplitud usando Rx(theta)
n_points = 60
theta_values = np.linspace(0, 4 * np.pi, n_points)
p1_sim = []

circuits = []
for theta in theta_values:
    qc = QuantumCircuit(1, 1)
    qc.rx(theta, 0)
    qc.measure(0, 0)
    circuits.append(qc)

# Batch en una sola llamada
shots = 2000
job = sampler.run(circuits, shots=shots)
for i, result in enumerate(job.result()):
    counts = result.data.c0.get_counts()
    p1_sim.append(counts.get('1', 0) / shots)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(theta_values / np.pi, p1_theory, 'b-', linewidth=2, label='Teórico')
ax.scatter(theta_values / np.pi, p1_sim, s=15, color='orange',
           alpha=0.7, label=f'Simulación ({shots} shots)')
ax.set_xlabel('θ / π', fontsize=12)
ax.set_ylabel('P(|1⟩)', fontsize=12)
ax.set_title('Oscilación Rabi: teoría vs simulación', fontsize=13)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Calibración del π-pulso con ajuste sinusoidal

En hardware real, el ángulo exacto del π-pulso se determina ajustando los datos a
una sinusoide A·sin²(f·θ + φ) + offset. El mínimo shots y el ruido de medición
introducen incertidumbre en la calibración.

In [ ]:
from scipy.optimize import curve_fit

def rabi_model(theta, A, f, phi, offset):
    return A * np.sin(f * theta / 2 + phi)**2 + offset

# Añadir ruido de medición realista
np.random.seed(42)
sigma = 0.02  # 2% de ruido de medición
p1_noisy = np.array(p1_theory[:n_points]) + np.random.normal(0, sigma, n_points)
p1_noisy = np.clip(p1_noisy, 0, 1)

# Ajuste de curva
try:
    popt, pcov = curve_fit(
        rabi_model, theta_values, p1_noisy,
        p0=[1.0, 1.0, 0.0, 0.0],
        bounds=([0.5, 0.5, -0.5, -0.1], [1.5, 1.5, 0.5, 0.1])
    )
    A_fit, f_fit, phi_fit, off_fit = popt
    perr = np.sqrt(np.diag(pcov))

    # π-pulso: argmax de sin²(f*theta/2) → f*theta/2 = π/2 → theta_pi = π/f
    theta_pi_fit = np.pi / f_fit

    print(f'Parámetros del ajuste:')
    print(f'  Amplitud A = {A_fit:.4f} ± {perr[0]:.4f}')
    print(f'  Frecuencia f = {f_fit:.4f} ± {perr[1]:.4f}')
    print(f'  θ_π calibrado = {theta_pi_fit:.4f} rad = {theta_pi_fit/np.pi:.4f}π')
    print(f'  θ_π teórico   = π = {np.pi:.4f} rad')
    print(f'  Error calibración: {abs(theta_pi_fit - np.pi)/np.pi * 100:.2f}%')
except RuntimeError as e:
    print(f'Ajuste fallido: {e}')

## 4. Efecto de la decoherencia en las oscilaciones

En hardware real, las oscilaciones de Rabi se **amortiguan** por la decoherencia T₂.
El modelo es: P(|1⟩) = A·e^{-t/T₂}·sin²(Ωt/2) + 0.5·(1 - e^{-t/T₂})

In [ ]:
def rabi_with_decoherence(theta, T2: float, Omega: float = 1.0) -> np.ndarray:
    """Oscilación Rabi amortiguada por T2. theta = tiempo en unidades de π."""
    t = theta / Omega  # tiempo
    decay = np.exp(-t / T2)
    return decay * np.sin(Omega * t / 2)**2 + 0.5 * (1 - decay)

thetas = np.linspace(0, 6 * np.pi, 300)
T2_values = [4*np.pi, 2*np.pi, np.pi, 0.5*np.pi]

fig, ax = plt.subplots(figsize=(10, 5))
for T2 in T2_values:
    p = rabi_with_decoherence(thetas, T2)
    ax.plot(thetas / np.pi, p, linewidth=2, label=f'T₂ = {T2/np.pi:.1f}π')

ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Límite mixto 0.5')
ax.set_xlabel('θ / π', fontsize=12)
ax.set_ylabel('P(|1⟩)', fontsize=12)
ax.set_title('Oscilaciones Rabi con decoherencia T₂', fontsize=13)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Calibración en la práctica: protocolo completo

El protocolo de calibración Rabi en hardware IBM sigue estos pasos:

1. **Barrido de amplitud**: medir P(|1⟩) para muchas amplitudes
2. **Ajuste de curva**: extraer Ω, A, T₂ del modelo amortiguado
3. **Calcular θ_π**: amplitud del π-pulso que maximiza P(|1⟩)
4. **Validar**: aplicar dos π-pulsos y verificar que regresa a |0⟩
5. **Repetir periódicamente**: el hardware deriva con el tiempo

In [ ]:
# Protocolo de validación: aplicar 2k π-pulsos, verificar regreso a |0⟩
print('Validación: 2k aplicaciones del π-pulso → debe volver a |0⟩')
print(f'{"k":<5} | {"2k puertas":<12} | P(|1⟩) esperada')
print('-' * 35)
for k in [1, 2, 3, 4, 5]:
    theta_total = 2 * k * np.pi  # 2k π-pulsos
    p1 = rabi_probability(theta_total)
    print(f'{k:<5} | {2*k:<12} | {p1:.6f}')

print('\nCalibración √X (π/2-pulso):')
for n_sq in [1, 2, 4]:
    theta = n_sq * np.pi / 2
    p1 = rabi_probability(theta)
    print(f'  {n_sq} × (π/2) = {n_sq}π/2: P(|1⟩) = {p1:.4f}')